In [ ]:
# Setup

import numpy as np
import pandas as pd

from blox2 import BLOX2Selector, RandomSelector, split_df_by_n_rows
from blox2.predictor import RidgePredictor, LassoPredictor, RandomForestPredictor, SVRPredictor

# predictor = RidgePredictor()
# predictor = LassoPredictor()
predictor = RandomForestPredictor(n_estimators=100)
# predictor = SVRPredictor()

observed_features, unchecked_features = split_df_by_n_rows(pd.read_csv("../data/zinc_dft/feature_list.csv", header=None), 10)
observed_values, unchecked_values = split_df_by_n_rows(pd.read_csv("../data/zinc_dft/properties.csv"), 10)

# observed_features = pd.read_csv("../data/test/feature_list_of_observed_data.csv", header=None)
# unchecked_features = pd.read_csv("../data/test/feature_list_of_unchecked_data.csv", header=None)
# observed_values = pd.read_csv("../data/test/properties_of_observed_data.csv")
# unchecked_values = pd.read_csv("../data/test/properties_of_unchecked_data.csv")

def get_true_value(id):
    return np.asarray(unchecked_values.iloc[id - len(observed_features)])

In [ ]:
# Loop

import time

n_iters = 1000
report_interval = 10
squared_sigma = 1

selector = BLOX2Selector(observed_features, observed_values, unchecked_features, predictor, squared_sigma=squared_sigma, normalize_features=False, n_obs_samples=None, normalize_values=True, n_chunks=512, use_distribution=False, compare_selection_time=False)
# selector = RandomSelector(observed_features, observed_values, unchecked_features)

t0 = time.perf_counter()
for i in range(min(n_iters, len(unchecked_features))):
    ids = selector.next_candidates(n=1)
    for id in ids:
        selector.observe(id, get_true_value(id))
    if (i+1) % report_interval == 0:
        print(f"{i+1} candidates suggested. Passed time: {time.perf_counter() - t0}")

In [ ]:
import csv, datetime, os
import matplotlib.pyplot as plt
import pandas as pd
from blox2 import calc_stein_discrepancy_trajectory

fixed_data_for_scaler = np.vstack([observed_values, unchecked_values]) # uses all property values
dir_name = "results/" + datetime.datetime.now().strftime("%m-%d_%H-%M")
os.makedirs(dir_name, exist_ok=True)

# Record results
observation_history = selector.make_observation_history()

with open(os.path.join(dir_name, "candidate_histories.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    for x in selector.candidate_id_history[selector.initial_n_obs:]:
        writer.writerow([x])
np.savetxt(os.path.join(dir_name, "initial_observed_properties.csv"), observation_history[:selector.initial_n_obs], delimiter=",",)
np.savetxt(os.path.join(dir_name, "observation_histories.csv"), observation_history[selector.initial_n_obs:], delimiter=",",)

# Record times

df = pd.DataFrame({
    "Selection": selector.passed_times_selection,
    "Train": selector.passed_times_train,
    "Pred": selector.passed_times_pred,
    "Total": selector.passed_times_total
})
df["Misc"] = df["Total"] - df["Selection"] - df["Train"] - df["Pred"]
df.to_csv(os.path.join(dir_name, "time_consumption.csv"), index=False)
ax = df.drop(columns="Total").plot()
ax.set_xlabel("Number of samplings")
ax.set_ylabel("Time consumption (sec)")
plt.tight_layout()
plt.savefig(os.path.join(dir_name, "time_consumption.png"))
plt.show()

# Plot and save Stein discrepancy trajectory with fixed scaling

sd_trajectory = calc_stein_discrepancy_trajectory(observation_history, scale=fixed_data_for_scaler, squared_sigma=squared_sigma)
plt.figure()
plt.xlabel("Number of samplings")
plt.ylabel("Stein discrepancy")
plt.plot(sd_trajectory[selector.initial_n_obs:])
plt.savefig(os.path.join(dir_name, f"stein_discrepancy_ss={squared_sigma}.png"))

np.savetxt(os.path.join(dir_name, f"stein_discrepancy_histories_ss={squared_sigma}.csv"), sd_trajectory[selector.initial_n_obs:], delimiter=",",)